# **Agente ReAct con Memoria Persistente — Notebook paso a paso**

Esta pre-entrega es distinta a las anteriores en el fondo: acá no armamos un flujo fijo (Supervisor → especialista → síntesis), sino un ciclo abierto donde el propio LLM decide, turno a turno, si necesita usar una herramienta o si ya puede responder. Ese ciclo (modelo → ¿tool? → herramienta → modelo → ...) es el patrón ReAct (Reasoning + Acting).

In [ ]:
!pip install -q -U \
langchain \
langchain-core \
langchain-google-genai \
langchain-openai \
langgraph \
langgraph-checkpoint-sqlite \
aiosqlite \
pydantic \
python-dotenv

print("\n" + "=" * 70)
print("⚠️  REINICIÁ EL ENTORNO DE EJECUCIÓN AHORA y seguí desde la próxima")
print("    celda, SIN volver a correr esta celda de instalación.")
print("=" * 70)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.

⚠️  R

**API key de Gemini**

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("🔑 GOOGLE_API_KEY (gratis en aistudio.google.com/apikey): ").strip()

🔑 GOOGLE_API_KEY (gratis en aistudio.google.com/apikey): ··········


In [ ]:
# ✅ Gemini queda como proveedor por defecto.
# Cambiá a True únicamente si querés usar OpenAI.
USAR_OPENAI = True

if USAR_OPENAI:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("🔑 OPENAI_API_KEY: ").strip()
    print("✅ Se usará OpenAI como proveedor del LLM.")
else:
    print("✅ Se usará Gemini como proveedor del LLM.")


##**"Base de datos" simulada**

Por qué dos tablas separadas: para forzar que el agente tenga que razonar en dos pasos — no le damos el cliente_id directo, solo el nombre. Así el LLM tiene que decidir por sí mismo: "primero necesito el ID, después los pedidos".

In [ ]:
CLIENTES_DB = {
    "ana garcía": 102,
    "juan pérez": 205,
    "maría lópez": 310,
}

PEDIDOS_DB = {
    102: {"pedidos": 3, "total": 14500},
    205: {"pedidos": 1, "total": 3200},
    310: {"pedidos": 5, "total": 27800},
}

Herramientas (@tool con docstrings descriptivos)

Qué error evita: "Descripciones Vagas" — el docstring de cada función es lo único que el LLM lee para decidir cuándo usarla. Un docstring genérico ("busca info") hace que el agente elija mal la herramienta o no la use cuando debería.

In [ ]:
from langchain_core.tools import tool


@tool
def buscar_cliente_por_nombre(nombre: str) -> str:
    """Busca el ID interno (cliente_id) de un cliente a partir de su nombre completo.
    Usar esta herramienta SIEMPRE que el usuario mencione un cliente por su nombre
    y necesites averiguar su cliente_id antes de poder consultar sus pedidos.
    Devuelve un mensaje de ERROR si el nombre no coincide exactamente con ningún
    cliente registrado."""
    clave = nombre.strip().lower()
    if clave not in CLIENTES_DB:
        return f"ERROR: no se encontró ningún cliente con el nombre '{nombre}'. Verificá que el nombre esté completo y bien escrito."
    return f"Cliente encontrado: '{nombre}' → cliente_id={CLIENTES_DB[clave]}"


@tool
def buscar_pedidos(cliente_id: int) -> str:
    """Busca la cantidad de pedidos y el monto total gastado por un cliente,
    dado su cliente_id NUMÉRICO (no su nombre — si solo tenés el nombre, primero
    usá buscar_cliente_por_nombre para obtener el ID).
    Devuelve un mensaje de ERROR si el cliente_id no existe en la base de datos."""
    if cliente_id not in PEDIDOS_DB:
        return f"ERROR: no existe ningún cliente con id={cliente_id}."
    datos = PEDIDOS_DB[cliente_id]
    return f"pedidos={datos['pedidos']}, total=${datos['total']}"


herramientas = [buscar_cliente_por_nombre, buscar_pedidos]

LLM vinculado a las herramientas

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)
llm_con_herramientas = llm.bind_tools(herramientas)

El grafo (nodo modelo + nodo herramientas)

In [ ]:
from typing import TypedDict, Sequence
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


async def nodo_modelo(state: MessagesState) -> dict:
    """Le pasa el historial completo al LLM y devuelve su respuesta (que puede
    incluir tool_calls o ser la respuesta final)."""
    respuesta = await llm_con_herramientas.ainvoke(state["messages"])
    return {"messages": [respuesta]}


grafo = StateGraph(MessagesState)  # hereda el reducer add_messages, como pide la consigna
grafo.add_node("modelo", nodo_modelo)
grafo.add_node("herramientas", ToolNode(herramientas, handle_tool_errors=True))

grafo.add_edge(START, "modelo")
grafo.add_conditional_edges(
    "modelo",
    tools_condition,  # mira si el último mensaje trae tool_calls
    {"tools": "herramientas", END: END},
)
grafo.add_edge("herramientas", "modelo")  # el ciclo: vuelve al modelo con el resultado de la tool

Este add_edge("herramientas", "modelo") es el "ciclo" del título de la consigna — a diferencia del grafo lineal de la Pre-entrega 6, acá el modelo puede volver a pasar por la herramienta las veces que necesite, no una sola vez.

Persistencia con AsyncSqliteSaver

Qué error evita: el "Estado Sucio" y la falta de "Resiliencia de Estado". Sin checkpointer, cada invoke() arranca de cero — con thread_id, LangGraph guarda el historial en un archivo SQLite y lo recupera automáticamente la próxima vez que uses el mismo thread_id.

In [ ]:
import json
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver


def extraer_texto(mensaje) -> str:
    """Normaliza el content de un mensaje (Gemini a veces devuelve una lista de bloques)."""
    contenido = mensaje.content
    if isinstance(contenido, str):
        return contenido
    if isinstance(contenido, list):
        return "".join(b.get("text", "") for b in contenido if isinstance(b, dict))
    return str(contenido)


def serializar_traza(mensajes) -> list:
    """Convierte los mensajes de LangGraph a un formato JSON simple, para dejar
    la traza ReAct como log dentro del repo (pedido explícito de la consigna)."""
    traza = []
    for m in mensajes:
        entrada = {"tipo": type(m).__name__, "contenido": extraer_texto(m)}
        if isinstance(m, AIMessage) and m.tool_calls:
            entrada["tool_calls"] = [
                {"nombre": tc["name"], "argumentos": tc["args"]} for tc in m.tool_calls
            ]
        if isinstance(m, ToolMessage):
            entrada["herramienta"] = m.name
        traza.append(entrada)
    return traza

Prueba: razonamiento multi-paso + memoria entre turnos

In [ ]:
CONFIG = {"configurable": {"thread_id": "conversacion-demo-1"}, "recursion_limit": 10}


async def main():
    async with AsyncSqliteSaver.from_conn_string("checkpoints.sqlite") as checkpointer:
        app = grafo.compile(checkpointer=checkpointer)

        print("=" * 80)
        print("🧑 Turno 1: '¿Cuántos pedidos tuvo Ana García y cuál fue el total?'")
        print("=" * 80)
        resultado1 = await app.ainvoke(
            {"messages": [HumanMessage(content="¿Cuántos pedidos tuvo Ana García y cuál fue el total?")]},
            config=CONFIG,
        )
        print("🤖", extraer_texto(resultado1["messages"][-1]))

        print("\n" + "=" * 80)
        print("🧑 Turno 2 (mismo thread_id): '¿Y Juan Pérez?'")
        print("=" * 80)
        resultado2 = await app.ainvoke(
            {"messages": [HumanMessage(content="¿Y Juan Pérez?")]},  # depende del contexto del turno 1
            config=CONFIG,
        )
        print("🤖", extraer_texto(resultado2["messages"][-1]))

        return resultado1, resultado2


resultado1, resultado2 = await main()

🧑 Turno 1: '¿Cuántos pedidos tuvo Ana García y cuál fue el total?'
🤖 Ana García tuvo **3 pedidos** con un monto total de **$14.500**.

🧑 Turno 2 (mismo thread_id): '¿Y Juan Pérez?'


GoogleRateLimitError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.8-flash\nPlease retry in 31.763905964s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTie

el LLM recibe la pregunta, no tiene el cliente_id de Ana García, así que llama a buscar_cliente_por_nombre (paso 1 de la herramienta), recibe el ID, y autónomamente decide llamar a buscar_pedidos con ese ID (paso 2) — dos invocaciones de herramientas para una sola pregunta, sin que nadie le haya dicho el orden.

Qué estás viendo en el Turno 2: "¿Y Juan Pérez?" no tiene sentido sin memoria — el modelo necesita recordar que la pregunta anterior era sobre pedidos y totales. Como usamos el mismo thread_id, el checkpointer le devuelve el historial completo antes de responder.

Prueba del "Ciclo de Retorno" (error → pedido de aclaración)

In [ ]:
async def probar_error():
    async with AsyncSqliteSaver.from_conn_string("checkpoints.sqlite") as checkpointer:
        app = grafo.compile(checkpointer=checkpointer)
        config_error = {"configurable": {"thread_id": "conversacion-error-1"}, "recursion_limit": 10}

        resultado = await app.ainvoke(
            {"messages": [HumanMessage(content="¿Cuántos pedidos tuvo el cliente Roberto Sánchez?")]},
            config=config_error,
        )
        print("🤖", extraer_texto(resultado["messages"][-1]))
        return resultado


resultado_error = await probar_error()

Como "Roberto Sánchez" no existe en CLIENTES_DB, buscar_cliente_por_nombre devuelve un ERROR:.... Con handle_tool_errors=True, ese error vuelve como un mensaje normal al LLM (no como una excepción que rompe el programa) — el modelo lo lee y, en vez de inventar un número, te va a pedir que confirmes el nombre. Eso es exactamente el criterio de "Ciclo de Retorno" de la consigna.

Guardar la traza ReAct como .json (entregable pedido)

In [ ]:
traza_completa = {
    "turno_1_multi_paso": serializar_traza(resultado1["messages"]),
    "turno_2_memoria": serializar_traza(resultado2["messages"]),
    "prueba_error_y_aclaracion": serializar_traza(resultado_error["messages"]),
}

with open("traza_ejecucion.json", "w", encoding="utf-8") as f:
    json.dump(traza_completa, f, ensure_ascii=False, indent=2)

print("✅ Traza guardada en traza_ejecucion.json")
print(json.dumps(traza_completa["turno_1_multi_paso"], ensure_ascii=False, indent=2))